<a href="https://colab.research.google.com/github/VemireddyBhavana/helpora--scholarship-and-discount-queries-/blob/main/Helpora_Scholarship_%26_Discount_Queries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building Helpora 🤖 — Self-Paced Notebook

### A scholarship & discount support agent · Agentic AI Workshop · NIAT

Welcome! In the workshop you saw Helpora being built live for **payment refund issues**. This notebook extends Helpora so you can build the **next version of the agent yourself, step by step** — every cell tells you what to do, what to expect, and what to check before moving on.

The story: a student, **Riya (S-7-043)**, says her **scholarship for this term has not been applied to her fee invoice**. You will build an agent that looks up her real scholarship eligibility and invoice records, decides what happened, and either answers her directly or files a task for a human to review the case.

What you’ll build, in order:

| Section                     | What you build                                                                              |
| --------------------------- | ------------------------------------------------------------------------------------------- |
| Setup                       | Get the notebook running + connect your API key                                             |
| A — The brain               | Connect to an LLM (the model)                                                               |
| B — The use case + database | Create real scholarship, invoice, and payment records and see why a bare model fails        |
| C — Tools                   | Give the agent the ability to read scholarship, invoice, and payment data and take actions  |
| D — Prompting               | Teach Helpora how to handle scholarship, discount, and refund questions clearly and kindly  |
| E — The agent loop          | Wire everything into a ReAct agent and watch it work on both refund and scholarship tickets |

⏱️ **Expect this to take 45–60 minutes** if you read everything carefully. Don’t rush — the reading is the learning.


## 🛠️ Setup Step 0 — Open this notebook in a runnable environment

You need a place where you can run Python notebook cells. Pick one of these:

### Option 1 (recommended, zero installation): Google Colab

* Go to [Google Colab](https://colab.research.google.com?utm_source=chatgpt.com)
* Click **File → Upload notebook** and upload this `.ipynb` file
* That’s it — you can run cells right away

### Option 2: VS Code

* Open this file in **VS Code**
* Install the **Jupyter extension** if prompted
* Select a **Python 3.10+ kernel** when asked

### Option 3: Local Jupyter

* If you have Python installed, run `pip install notebook` and then `jupyter notebook` in a terminal
* Open this file from the browser window that appears

### How to run a cell

Click on a cell and press **Shift + Enter** (or click the ▶️ play button). Run cells **top to bottom, in order** — later cells depend on earlier ones.

✅ **Check before continuing:** you can see this text in a notebook interface and there’s a code cell below you can click on.


## 🛠️ Setup Step 1 — Get your OpenRouter API key

Helpora’s “brain” is an LLM that we access through **[OpenRouter](https://openrouter.ai?utm_source=chatgpt.com)** — a service that gives one API for many models.

* Go to [OpenRouter](https://openrouter.ai?utm_source=chatgpt.com) and sign up (Google login works)
* Open **Keys** under your profile → click **Create Key**
* Give it any name and copy the key — it will look like `sk-or-v1-...`
* Keep it somewhere handy; you’ll paste it into the next cell

⚠️ **Treat the key like a password** — don’t share it or post screenshots of it.


## 🛠️ Setup Step 2 — Install the libraries

▶️ Run the cell below. It installs **LangChain**, the framework we’ll use to build Helpora’s scholarship, discount, and refund support agent.

We pin exact versions so everyone runs the **same LangChain setup** — the agent API has changed recently, and pinning avoids version surprises during the workshop.

**Expected output:** a line saying **Installed.**
It may take **30–60 seconds**. If you see a red pip **resolver warning**, it’s safe to ignore.


In [ ]:
!pip install -q langchain==1.3.11 langchain-core==1.4.8 langchain-openai==1.3.3 python-dotenv
print("Installed. If you see a pip resolver warning, it's safe to ignore.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.4 MB/s eta 0:00:00
Installed. If you see a pip resolver warning, it's safe to ignore.


## 🛠️ Setup Step 3 — Connect your API key

▶️ Run the cell below. It will show a small input box — paste your **OpenRouter** key there and press **Enter**. The key stays hidden, like a password field.

If you’re running the notebook locally and prefer a `.env` file, create a file named `.env` next to this notebook containing one line:

`OPENROUTER_KEY=your-key`

If that file exists, the cell will pick it up automatically and skip the input box.

**Expected output:**
`Key loaded (ends with ...xxxx).`


In [ ]:
from dotenv import load_dotenv
import os
from getpass import getpass

load_dotenv()
api_key = os.environ.get("OPENROUTER_KEY")

if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENROUTER_KEY")
    except Exception:
        pass

if not api_key:
    api_key = getpass("Paste your OpenRouter API key and press Enter: ").strip()

if not api_key:
    raise ValueError("No key entered. Re-run this cell and paste your OpenRouter key.")

print(f"Key loaded (ends with ...{api_key[-4:]}).")

Key loaded (ends with ...871a).


## A — The brain (the model)

An agent’s brain is just an **LLM**. We’ll use a fast, low-cost model (**gpt-4o-mini**) through **OpenRouter**, wrapped in LangChain’s `ChatOpenAI` class.

Two things to notice in the code:

* `model=` → which brain we’re renting
* `base_url=` → we point LangChain at **OpenRouter** instead of calling OpenAI directly

▶️ Run the cell below.

**Expected output:** `Model connected.`


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)
print("Model connected.")

Model connected.


## A-1 · Quick test — does the brain answer?

▶️ Run the cell below. We’ll send the model one tiny message just to confirm that Helpora’s brain is connected correctly.

**Expected output:** the single word **READY**.

❌ If you get a **401** or authentication error, your API key is missing, incorrect, or wasn’t loaded properly — re-run **Setup Step 3**.
❌ If you get a **402** or credits error, your **OpenRouter** account may not have enough credits available.


In [ ]:
print(llm.invoke("Reply with just the word: READY").content)

READY


## B — The use case + a real database

Real agents don’t work off text hard-coded in a prompt — they read from **real data**. In the refund notebook, Helpora only needed access to a small database of student payments. In this notebook, Helpora needs a broader view of the student’s fee situation.

To answer scholarship and discount questions properly, Helpora will read from a small **SQLite database** containing:

* **payment records** → for refund and duplicate-charge cases
* **scholarship eligibility records** → to check whether a student qualifies for a fee waiver
* **invoice records** → to check whether that scholarship or discount was actually applied to the student’s fee bill

Think of that database as the agent’s **long-term memory**: facts that live on “disk” and get loaded only when needed.


## B-1 · Create the demo Helpora database

▶️ Run the cell below. It creates a file called **`helpora.db`** with the records Helpora needs to answer **scholarship and discount-related support questions**.

In this notebook, the key case is **Riya (S-7-043)**. Riya scored **9.0 CGPA**, which makes her eligible for the **minimum 20% scholarship fee waiver**. But when her **Semester 3 invoice** arrived, no scholarship had been applied. Helpora must read the real records, notice that mismatch, and decide what to do next.

To do that, the database includes:

* **scholarship eligibility records** → to check whether a student qualifies for a scholarship or fee waiver
* **invoice records** → to verify whether the scholarship was actually applied to the student’s current term fee invoice
* **payment records** → so Helpora can continue to support refund-related questions as well, if needed

👀 Before running the cell, look at the sample data and notice that **Riya is eligible for a scholarship, but her invoice shows that no scholarship has been applied**. That is the issue Helpora must catch.

**Expected output:**
`helpora.db created with scholarship, invoice, and payment records.`


In [ ]:
import sqlite3

conn = sqlite3.connect("helpora.db")
cur = conn.cursor()

# Remove old tables if they exist
cur.execute("DROP TABLE IF EXISTS scholarships")
cur.execute("DROP TABLE IF EXISTS invoices")
cur.execute("DROP TABLE IF EXISTS payments")

# -------------------------
# 1) Scholarships table
# -------------------------
cur.execute("""
    CREATE TABLE scholarships (
        student_id TEXT,
        student_name TEXT,
        cgpa REAL,
        scholarship_percent INTEGER,
        status TEXT
    )
""")

cur.executemany(
    "INSERT INTO scholarships VALUES (?,?,?,?,?)",
    [
        ("S-7-043", "Riya Sharma", 9.0, 20, "eligible"),
        ("S-7-118", "Meera Nair", 8.8, 20, "eligible"),
        ("S-7-120", "Aarav Singh", 7.2, 0, "not_eligible")
    ]
)

# -------------------------
# 2) Invoices table
# -------------------------
cur.execute("""
    CREATE TABLE invoices (
        invoice_id TEXT,
        student_id TEXT,
        term TEXT,
        base_fee INTEGER,
        scholarship_applied_percent INTEGER,
        final_fee INTEGER,
        status TEXT
    )
""")

cur.executemany(
    "INSERT INTO invoices VALUES (?,?,?,?,?,?,?)",
    [
        # Riya: scholarship missing even though eligible
        ("INV-3001", "S-7-043", "Semester 3", 20000, 0, 20000, "issued"),

        # Meera: scholarship correctly applied
        ("INV-3002", "S-7-118", "Semester 3", 20000, 20, 16000, "issued"),

        # Aarav: not eligible, so no scholarship applied
        ("INV-3003", "S-7-120", "Semester 3", 20000, 0, 20000, "issued")
    ]
)

# -------------------------
# 3) Payments table
# Optional: keep this so refund support still works too
# -------------------------
cur.execute("""
    CREATE TABLE payments (
        payment_id TEXT,
        student_id TEXT,
        student_name TEXT,
        amount INTEGER,
        description TEXT,
        date TEXT
    )
""")

cur.executemany(
    "INSERT INTO payments VALUES (?,?,?,?,?,?)",
    [
        ("PAY-1001", "S-7-042", "Aditya Kumar", 15000, "Course fee - Semester 1", "2026-05-03"),
        ("PAY-1002", "S-7-042", "Aditya Kumar", 15000, "Course fee - Semester 1", "2026-05-03"),  # duplicate
        ("PAY-1003", "S-7-118", "Meera Nair", 15000, "Course fee - Semester 3", "2026-05-04"),
        ("PAY-1004", "S-7-120", "Aarav Singh", 15000, "Course fee - Semester 3", "2026-05-05")
    ]
)

conn.commit()
conn.close()

print("helpora.db created with scholarship, invoice, and payment records.")

helpora.db created with scholarship, invoice, and payment records.


## B-2 · The ticket we’re solving

This is the support ticket **Riya** raised. ▶️ Run the cell below to store it in a variable — we’ll reuse it throughout the notebook as Helpora learns how to handle scholarship and discount questions.


In [ ]:
TICKET = "I think my scholarship for this term hasn't been applied to my fee invoice. My student ID is S-7-043. — Riya"
print(TICKET)

I think my scholarship for this term hasn't been applied to my fee invoice. My student ID is S-7-043. — Riya


## B-3 · First try: NO tools — watch the model struggle

Before giving the agent anything special, let’s see what the bare model does with **Riya’s ticket**. At this point, Helpora has **no database access and no tools** — it only has the model from Section A.

▶️ Run the cell below, then read the model’s reply carefully and ask yourself:

* Did it actually check **Riya’s scholarship eligibility or invoice records**?
  *(It can’t — it has no access to the database.)*

* Did it do anything about the problem, like verify whether a scholarship was missed or file a task for review?
  *(It can’t — it has no actions.)*


In [ ]:
NO_TOOLS_PROMPT = f"""You are Helpora, a scholarship and fee-support agent.
Resolve this student's ticket. If you need to check their actual scholarship
eligibility or invoice records to answer, be honest about what you can and cannot access.

Ticket: {TICKET}"""

print(llm.invoke(NO_TOOLS_PROMPT).content)

Hello Riya,

Thank you for reaching out regarding your scholarship application. Unfortunately, I don't have access to your specific scholarship eligibility or invoice records directly. However, I can guide you on the best steps to take to resolve this issue.

1. **Check Your Scholarship Award Notification**: Make sure you have received official notification of your scholarship for this term. This could be an email or a letter from the scholarship office.

2. **Contact the Financial Aid Office**: I recommend reaching out to your institution's financial aid office directly. They can confirm whether your scholarship has been applied to your account and help address any discrepancies.

3. **Review Your Invoice**: Look over your fee invoice carefully. Ensure that the scholarship discount should have been reflected based on the terms provided in your scholarship notification.

If you have any further questions or need more assistance, please feel free to ask. 

Best regards,  
Helpora


What just happened? The model has no way to reach payments.db — so it either guesses or honestly admits it can't check the records. It can talk, but it can't act.

(The exact reply is model-dependent — run the cell 2–3 times and see how it varies.)

That gap is exactly what tools fill. Next, we give Helpora two tools — one to read the database, one to file a task — and watch the same ticket play out very differently.

C — Tools

A tool is just a Python function the agent is allowed to call. Tools are how Helpora moves beyond “just talking” and starts reading real records or taking real actions.

In this notebook, Helpora gets four tools:

lookup_scholarship(student_id) — reads the student’s scholarship eligibility record
lookup_invoice(student_id) — reads the student’s fee invoice record
lookup_payments(student_id) — reads payment history, so refund questions can still be supported

C-1 · Write the tool functions

▶️ Run the cell below. It defines all four functions and then calls a few of them directly so you can test them yourself before the agent starts using them.

Expected output:

A scholarship record for Riya showing that she is eligible for a 20% scholarship

An invoice record for Riya showing that 0% scholarship has been applied

Created TASK-001: Test task

In [ ]:
import sqlite3, json

# -------------------------
# Tool 1: Scholarship lookup
# -------------------------
def lookup_scholarship(student_id: str) -> str:
    """Return scholarship eligibility details for a student as JSON."""
    conn = sqlite3.connect("helpora.db")
    row = conn.execute(
        """
        SELECT student_id, student_name, cgpa, scholarship_percent, status
        FROM scholarships
        WHERE student_id = ?
        """,
        (student_id.strip(),)
    ).fetchone()
    conn.close()

    if not row:
        return f"No scholarship record found for {student_id}."

    return json.dumps({
        "student_id": row[0],
        "student_name": row[1],
        "cgpa": row[2],
        "scholarship_percent": row[3],
        "status": row[4]
    })

# -------------------------
# Tool 2: Invoice lookup
# -------------------------
def lookup_invoice(student_id: str) -> str:
    """Return the latest invoice details for a student as JSON."""
    conn = sqlite3.connect("helpora.db")
    row = conn.execute(
        """
        SELECT invoice_id, student_id, term, base_fee,
               scholarship_applied_percent, final_fee, status
        FROM invoices
        WHERE student_id = ?
        """,
        (student_id.strip(),)
    ).fetchone()
    conn.close()

    if not row:
        return f"No invoice found for {student_id}."

    return json.dumps({
        "invoice_id": row[0],
        "student_id": row[1],
        "term": row[2],
        "base_fee": row[3],
        "scholarship_applied_percent": row[4],
        "final_fee": row[5],
        "status": row[6]
    })

# -------------------------
# Tool 3: Payments lookup
# Optional but useful so refund support still works
# -------------------------
def lookup_payments(student_id: str) -> str:
    """Return all payment rows for a student_id as JSON."""
    conn = sqlite3.connect("helpora.db")
    rows = conn.execute(
        """
        SELECT payment_id, amount, description, date
        FROM payments
        WHERE student_id = ?
        """,
        (student_id.strip(),)
    ).fetchall()
    conn.close()

    if not rows:
        return f"No payments found for {student_id}."

    return json.dumps([
        {
            "payment_id": r[0],
            "amount": r[1],
            "description": r[2],
            "date": r[3]
        }
        for r in rows
    ])

# -------------------------
# Tool 4: Create task
# -------------------------
TASK_BOARD = []

def create_task(summary: str) -> str:
    """Add a follow-up task to the board for a human to handle."""
    task_id = f"TASK-{len(TASK_BOARD) + 1:03d}"
    TASK_BOARD.append({"id": task_id, "summary": summary})
    return f"Created {task_id}: {summary}"

# -------------------------
# Quick sanity checks
# -------------------------
print(lookup_scholarship("S-7-043"))
print(lookup_invoice("S-7-043"))
print(create_task("Test task"))

{"student_id": "S-7-043", "student_name": "Riya Sharma", "cgpa": 9.0, "scholarship_percent": 20, "status": "eligible"}
{"invoice_id": "INV-3001", "student_id": "S-7-043", "term": "Semester 3", "base_fee": 20000, "scholarship_applied_percent": 0, "final_fee": 20000, "status": "issued"}
Created TASK-001: Test task


## C-2 · Wrap the functions as LangChain tools

Right now these are ordinary Python functions — the model doesn’t know they exist yet. `Tool(name=, func=, description=)` turns each function into something the agent can **discover and call on its own**.

🔑 **The description matters a lot.** It’s the main clue the model uses when deciding **which tool to call**, **when to call it**, and **what input to pass**. So each description should clearly explain:

* what the tool does
* what kind of student record it reads or action it takes
* what input format it expects

▶️ Run the cell below.

**Expected output:**
`Tools ready: ['lookup_scholarship', 'lookup_invoice', 'lookup_payments', 'create_task']`


In [ ]:
from langchain_core.tools import Tool

scholarship_tool = Tool(
    name="lookup_scholarship",
    func=lookup_scholarship,
    description="Look up scholarship eligibility details for a student. Input: the student_id, e.g. 'S-7-043'. Returns scholarship eligibility, CGPA, and scholarship percentage as JSON."
)

invoice_tool = Tool(
    name="lookup_invoice",
    func=lookup_invoice,
    description="Look up the latest fee invoice for a student. Input: the student_id, e.g. 'S-7-043'. Returns invoice details including scholarship applied percentage as JSON."
)

payments_tool = Tool(
    name="lookup_payments",
    func=lookup_payments,
    description="Look up payment records for a student. Input: the student_id, e.g. 'S-7-042'. Returns payment records as JSON."
)

task_tool = Tool(
    name="create_task",
    func=create_task,
    description="File a follow-up task for a human. Input: a one-line summary of what needs to be checked or fixed."
)

tools = [scholarship_tool, invoice_tool, payments_tool, task_tool]
print("Tools ready:", [t.name for t in tools])

Tools ready: ['lookup_scholarship', 'lookup_invoice', 'lookup_payments', 'create_task']


## D — Prompting: same model, very different behaviour

Before wiring in the tools, there’s one more thing that matters a lot: **the prompt**.

Even with the same model, the way we **describe the role, task, and expected behavior** can change the quality of the response quite a bit. A vague prompt often produces a vague answer. A more structured prompt gives the model a clearer job to do.

### D-1 · A vague prompt

▶️ Run the cell below — it uses a generic **“you are a support assistant”** prompt. Read the reply and notice what feels weak, generic, or incomplete about it, especially for a case as important as **Riya’s missing scholarship**.


In [ ]:
VAGUE_PROMPT = f"""You are a customer support assistant for an educational institution.
A student has raised a support ticket. Read it carefully and write a helpful,
professional response that addresses their concern.

Ticket: {TICKET}"""

print(llm.invoke(VAGUE_PROMPT).content)

Subject: Re: Inquiry About Scholarship Application to Fee Invoice

Dear Riya,

Thank you for reaching out to us regarding your scholarship application for this term. I understand your concern about it not reflecting on your fee invoice.

To assist you further, I will look into your account to verify the status of your scholarship for this term. Please allow me a moment to check the details, and I will get back to you shortly with an update.

In the meantime, if you have any additional information or documentation regarding your scholarship that you think would be helpful, please feel free to share it with me.

Thank you for your patience, and I’ll be in touch soon.

Best regards,

[Your Name]  
Customer Support Team  
[Educational Institution Name]  
[Contact Information]  



## D-2 · A structured prompt

Now we give the model a **specific role**, a **clear step-by-step procedure**, and explicit instructions for **how to reply to the student**.

▶️ Run the cell below, then compare it with the vague prompt from **D-1**.

As you read the response, ask yourself:

* What changed in the **tone** and **structure** of the reply?
* Which part of the prompt seems to have caused each change — the **role**, the **numbered steps**, or the instruction to **reply to the student by name**?

💡 Keep this prompt in mind — in **Section E** we’ll hand this same logic to the agent, and once tools are connected it will be able to **actually perform steps 1–3** instead of just talking about them.


In [ ]:
STRUCTURED_PROMPT = f"""You are Helpora, a friendly campus fee-support agent who replies
directly to students.

For each ticket:
1. Identify whether the issue is about a refund / duplicate payment or a scholarship / discount.
2. If it is a scholarship or discount issue, check the student's scholarship eligibility and invoice details.
3. If it is a refund issue, check the student's payment records.
4. If a refund, scholarship correction, or manual review is needed, file a task for a human to handle it.
5. Finally, write a short, warm reply addressed to the student by name — tell them
   what you found and what happens next. This reply is what the student receives.

Ticket: {TICKET}
"""

print(llm.invoke(STRUCTURED_PROMPT).content)

Hello Riya,

Thank you for reaching out! I understand your concern regarding the scholarship not being applied to your fee invoice. I'll take a moment to check your scholarship eligibility and invoice details to ensure everything is in order.

After reviewing your account, it appears that your scholarship is eligible, but it may not have been applied correctly to your invoice. I will file a task for a human representative to conduct a manual review and ensure that your scholarship is applied properly.

You should receive an update shortly regarding the resolution. Thank you for your patience!

Warm regards,  
Helpora


## 🤔 Think about it

Even with the better prompt, the model still couldn’t **actually** look up scholarship records, invoice details, or payment history — and it couldn’t file a task for a human either. It can only **talk about** what it would do.

That’s the key idea of this notebook:

* **Prompting improves the thinking**
* **Tools enable the doing**

In the next section, we combine both. Helpora will keep the clearer instructions from the structured prompt, but now it will also be able to **read real student records** and **take follow-up actions** when needed.

💡 **Bonus question:** what other prompting techniques do you know — few-shot examples, role-play, or asking the model to reason step by step? Try editing the prompt above and re-running it to see how the reply changes.


## E — The ReAct agent: the loop that runs itself

Until now, **we** were the ones calling the tools by hand in **Section C**. A real agent works differently: it decides for itself **when** to use a tool, **which** tool to use, and **what to do next** based on the result.

The loop looks like this:

**Reason → Act (call a tool) → Observe (read the result) → Reason again → … → final reply**

LangChain’s `create_agent(...)` builds that loop for us.

## E-1 · Build the agent

▶️ Run the cell below. Notice that the agent is built from the same three ingredients you’ve already created:

* `model=llm` → **Section A** (the brain)
* `tools=tools` → **Section C** (the hands: scholarship lookup, invoice lookup, payment lookup, and task creation)
* `system_prompt=` → **Section D** (the instructions the agent should follow)

This time, the prompt is written as a **general system prompt** without the ticket itself inside it. We’ll pass the actual student ticket in when we run the agent.

**Expected output:**
`Agent built.`


In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """You are Helpora, a friendly campus fee-support agent who replies
directly to students.

Your job is to handle two kinds of student tickets:
1. Refund / duplicate payment issues
2. Scholarship / discount application issues

For each ticket:
1. First identify whether the issue is about a refund or a scholarship / discount.
2. If it is a scholarship or discount issue:
   - Look up the student's scholarship eligibility.
   - Look up the student's invoice.
   - Decide whether the scholarship or discount should have been applied.
3. If it is a refund or duplicate-payment issue:
   - Look up the student's payment records.
   - Decide whether there is a duplicate or incorrect charge.
4. If a refund, scholarship correction, or manual review is needed, file a task for a human.
5. Finally, write a short, warm reply addressed to the student by name explaining what you found and what happens next.

Important:
- Never invent records you have not checked with tools.
- You cannot directly move money or edit invoices yourself.
- Be especially gentle when the answer is "no" or when the student is not eligible.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)
print("Agent built.")

Agent built.


## E-2 · Run it — and watch the loop live

The cell below runs the agent on **Riya’s scholarship ticket** and prints every move it makes:

* **[REASON → ACT]** — the agent decided to call a tool (and with what input)
* **[OBSERVE]** — the result the tool returned to the agent
* **[REPLY TO STUDENT]** — the final message once the agent decides it’s done

▶️ Run the cell, then check the output against this checklist:

* Did the agent identify this as a **scholarship / discount issue** rather than a refund issue?
* Did it call **`lookup_scholarship`** with `S-7-043`?
* Did it call **`lookup_invoice`** with `S-7-043`?
* Did it notice that **Riya is eligible for a 20% scholarship** but her invoice shows **0% applied**?
* Did it call **`create_task`** to file a human review / correction task?
* Is there a task on the task board at the end?
* Does the final reply address **Riya by name** and explain what happens next?

Nobody told the agent exactly which tools to call or in what order — it decided each step for itself. That’s the difference between a **chatbot** and an **agent**.


In [ ]:
def run_agent(ticket: str):
    print(f"[GOAL]     {ticket}\n")
    final_reply = ""

    for step in agent.stream({"messages": [("user", ticket)]}, stream_mode="updates"):
        for node, payload in step.items():
            for msg in payload.get("messages", []):
                tool_calls = getattr(msg, "tool_calls", None)
                content = getattr(msg, "content", "")

                if node in ("model", "agent"):
                    if tool_calls:
                        for tc in tool_calls:
                            print(f"[REASON → ACT]  decided to call {tc['name']}({tc['args']})")
                    elif content:
                        final_reply = content

                elif node == "tools":
                    print(f"[OBSERVE]      {str(content)[:150]}")

    print(f"\n[REPLY TO STUDENT] {final_reply}")
    print("Task board:", TASK_BOARD)

run_agent(TICKET)

[GOAL]     I think my scholarship for this term hasn't been applied to my fee invoice. My student ID is S-7-043. — Riya

[REASON → ACT]  decided to call lookup_scholarship({'__arg1': 'S-7-043'})
[REASON → ACT]  decided to call lookup_invoice({'__arg1': 'S-7-043'})
[OBSERVE]      {"student_id": "S-7-043", "student_name": "Riya Sharma", "cgpa": 9.0, "scholarship_percent": 20, "status": "eligible"}
[OBSERVE]      {"invoice_id": "INV-3001", "student_id": "S-7-043", "term": "Semester 3", "base_fee": 20000, "scholarship_applied_percent": 0, "final_fee": 20000, "st
[REASON → ACT]  decided to call create_task({'__arg1': 'Scholarship not applied for Riya Sharma (S-7-043). She is eligible for a 20% scholarship.'})
[OBSERVE]      Created TASK-002: Scholarship not applied for Riya Sharma (S-7-043). She is eligible for a 20% scholarship.

[REPLY TO STUDENT] Hi Riya,

I checked your scholarship eligibility and found that you are eligible for a 20% scholarship, given your CGPA of 9.0. However, it loo

## 🎉 That’s a working agent!

You’ve now extended Helpora from a **refund-only support agent** into a broader **campus fee-support agent** that can also handle **scholarship and discount queries**.

Helpora now has:

* **a model** to reason about the student’s issue
* **a real database** with scholarship, invoice, and payment records
* **tools** to look up those records and create follow-up tasks
* **a structured prompt** that tells it how to handle different types of student tickets
* **an agent loop** that decides which tool to call, reads the result, and keeps going until it can reply

In Riya’s case, Helpora should be able to:

1. Recognize that the ticket is about a **scholarship issue**
2. Check **scholarship eligibility**
3. Check the **invoice**
4. Notice that the scholarship appears to be **missing**
5. File a **human review / correction task**
6. Reply to Riya warmly with what happens next

---

## 🧪 Try it yourself before the next session

Now that the main scholarship flow works, try a few more cases and see how the agent behaves.

### 1) The no-problem scholarship case

Change the ticket to **Meera** and re-run **E-2**:

```python
TICKET = "Can you check whether my scholarship was applied correctly? My student ID is S-7-118. — Meera"
```

Meera is eligible, and her invoice already shows the scholarship applied correctly.
Watch whether the agent checks the records and **reassures her without filing a task**.

---

### 2) The not-eligible case

Try **Aarav**:

```python
TICKET = "My invoice doesn't show any scholarship. My student ID is S-7-120. — Aarav"
```

Aarav’s CGPA is below the scholarship cutoff, so Helpora should explain gently that he is **not currently eligible**.

---

### 3) A refund question still working

If you kept the `payments` table and `lookup_payments` tool, try a refund ticket too:

```python
TICKET = "I think I was charged twice for my course fee. My student ID is S-7-042. — Aditya"
```

This checks whether Helpora can still support the **original refund flow** while also handling scholarship tickets.

---

### 4) An unknown student

Try a student ID that doesn’t exist in the database:

```python
TICKET = "I think my scholarship is missing. My student ID is S-9-999. — Unknown Student"
```

How does the agent respond when it can’t find scholarship or invoice records?
Does it stay honest and helpful?

---

### 5) Break a tool description on purpose

In **C-2**, change one tool description to something vague like `"a tool"` and re-run the notebook from **C-2 onward**.
Does the agent still pick the right tool? This is a good way to see why **tool descriptions matter** so much.

---

## 🚀 What this notebook teaches

This notebook uses the same core pattern as the original Helpora refund notebook:

**Model + Database + Tools + Prompt + Agent Loop**

That pattern is the heart of many practical AI agents. The use case changed from **duplicate-payment refunds** to **scholarship and discount support**, but the building blocks stayed the same.

If you get stuck, post the **error message** and the **cell where it happened** in the group so someone can help you debug it.
